In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from skimage import morphology, segmentation
from scipy.ndimage import rotate
from skimage import segmentation, morphology
from matplotlib.colors import hsv_to_rgb
import Chain_Analysis_Functions as caf

## Step 1: Read in the Image and create the initial mask

In [ ]:
original_image_name = '53-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.2_20mT', original_image_name)
mask = caf.create_binary_mask(image_path, brightness=1.0, contrast=1.0, saturation=1.0,
                           temperature=0, R_min=0, G_min=0, B_min=30, V_min=0.1,
                           method="adaptive", adaptive_block_size=10, adaptive_offset=0.01)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 8))

ax1.imshow(mask, cmap='gray')
ax1.set_title("Original Image")
ax1.axis("off")

# Top-left corner (works as-is)
ax2.imshow(mask[0:520, 0:520], cmap='gray')
ax2.set_title("Top left corner")
ax2.axis("off")

# Bottom-right corner — corrected slicing
ax3.imshow(mask[-520:, -520:], cmap='gray')
ax3.set_title("Bottom right corner")
ax3.axis("off")

plt.tight_layout()
plt.show()

## Step 2: Rotate the mask such that the particles align into perfect rows and columns

#### Identify the appropriate rotation angle

In [ ]:
rot_angle_mask = morphology.remove_small_objects(mask, min_size= 100) #Remove noise to keep it from interfering
rot_angle_mask, angle = caf.rotate_mask_until_balanced(rot_angle_mask, angle_step=0.2, tolerance=0.005, min_step=0.01, max_angle=5, search_rows = 50, search_columns = 3000)
print(angle)

In [ ]:
angle = 0.15947265625

#### Apply the rotation to the actual mask and confirm

In [ ]:
mask_rotated = rotate(mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(mask_rotated)

#### Update the mask

In [ ]:
mask = mask_rotated
del mask_rotated

## Step 3: Identify the reference particle bounding box

In [ ]:
ref_bbox = (34,25,237,228)
caf.show_reference(mask, ref_bbox)

## Step 4: Ensure reference crop is filled properly with set parameters

In [ ]:
ref_crop = caf.fill_reference(mask,ref_bbox, min_size = 100, pad = 0)
caf.display_mask(ref_crop,5,5)

## Step 5: Ensure Last Data is Properly Identified

In [ ]:
last_x, last_y = caf.check_last_row_and_column(mask, min_size = 100, last_row = 200, last_column = 200, plot = True)

## Step 6: Ensure array and particles are properly captured

### Check X

#### Modify the dx_offset and stagger_x parameters until it matches up

In [ ]:
dx_offset = 1
stagger_x = True
stagger_x_frequency = 2

checking = True
num_rows = 1
num_cols = 25
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

### Check Y

#### Modify the dy_offset and stagger_y parameters until it matches up

In [ ]:
dy_offset = 1
stagger_y = True
stagger_y_frequency = 2

checking = True
num_rows = 25
num_cols = 1
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

## Step 7: Apply all parameters and pull out particles from debris

In [ ]:
checking = False
erode_pixels = 2
num_rows = 25
num_cols = 25
particle_mask, debris_mask = caf.extract_particles_and_debris(mask, ref_bbox, ref_crop, min_size = 100, pad=10, max_gap = 10, dx_offset = dx_offset, dy_offset = 0,
                                                         stagger_x = stagger_x, stagger_y = False, erode_pixels = 0, checking = checking,
                                                         check_last_row =  200, check_last_column = 200, num_rows = num_rows, num_cols = num_cols)

In [ ]:
caf.display_mask(particle_mask, 10, 10)

In [ ]:
caf.display_mask(particle_mask[0:250,0:250], 10, 10)

In [ ]:
caf.display_mask(debris_mask0,10,10)

In [ ]:
caf.save_mask(particle_mask, image_path, '_0.png')
caf.save_mask(debris_mask, image_path,'_debris_0.png')

## Step 7: Identify bounds where particles are missing on the final wafer

### Step 7a: Load in cut wafer image and convert to a mask

In [ ]:
original_image_name = '0.2wt_strong_al_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.2_20mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

In [ ]:
cut_mask = caf.cropped_image_to_mask(cropped_image, method="otsu")

### Step 7b: Rotate the Mask Roughly

In [ ]:
angle = 178
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated,10,10)

### Step 7c: Perform Initial Crop

In [ ]:
ymin = 95
xmin = 790
cut_mask = cut_mask_rotated[ymin:, xmin:]
caf.display_mask(cut_mask,10,10)

### Step 7d: Finely Rotate the Mask

In [ ]:
rot_angle_mask = ~cut_mask
rot_angle_cut_mask = morphology.remove_small_objects(rot_angle_mask, min_size= 200) #Remove noise to keep it from interfering
caf.display_mask(rot_angle_cut_mask, 10,10)
rot_angle_cut_mask, cut_angle = caf.rotate_mask_until_balanced(rot_angle_cut_mask, angle_step=0.2, tolerance=0.005, min_step=0.01, max_angle=5, search_rows = 10, search_columns = 300)
print(cut_angle)
caf.display_mask(rot_angle_cut_mask, 10,10)

In [ ]:
angle = 0.029632201755674267
cut_mask_rotated = rotate(cut_mask, angle, reshape=False, order=0, mode='constant', cval=0)
caf.display_mask(cut_mask_rotated, 10, 10)

### Step 7e: Identify the appropriate bounds in particle mask (roughly)

In [ ]:
plus_x = 1220
plus_y = 1450
match_mask = particle_mask[:np.shape(cut_mask_rotated)[0]+plus_y, :np.shape(cut_mask_rotated)[1]+plus_x]
caf.display_mask(match_mask,10, 10)

In [ ]:
modify_rows_left = 7
modify_rows_right = -380
modify_cols_top = 7
modify_cols_bottom = -100
cut_mask2 =  caf.match_masks(cut_mask_rotated, match_mask, modify_rows_left, modify_rows_right, modify_cols_top, modify_cols_bottom)

In [ ]:
caf.display_mask(cut_mask2, 10, 10)

### Step 7f: Add rows and columns to match with the particle mask

In [ ]:
cut_mask3 = caf.add_rows_to_match(cut_mask2, particle_mask)

### Step 7g: Clean the mask to separate the area that has no particles

In [ ]:
cut_mask4 = caf.isolate_empty_space(cut_mask3, remove_particle_size = 100000, small_object_size = 2000, max_column_gap1 = 50, max_row_gap = 200, max_filled_col_gap = 500,
                        max_column_gap2 = 50, enhance_large_gap_size =200000, large_hole_threshold = 2000000, plot = True)

## Step 8: Filter particle mask using blank space from cut mask

In [ ]:
caf.display_mask(particle_mask,10,10)

In [ ]:
caf.display_mask(cut_mask3,10,10)

In [ ]:
original_image_name = '0.2wt_strong_al_n1_VSM.jpg'
wafer_path = os.path.join(os.getcwd(), '0.2_20mT', original_image_name)
cropped_image = caf.show_original_image(wafer_path, x_size = 10, y_size = 10)

### Isolate region of the debris and particle masks using the cut mask

In [ ]:
filtered_particle_mask = particle_mask & cut_mask4
caf.display_mask(filtered_particle_mask,10,10)
filtered_particle_mask_cropped = filtered_particle_mask[:3450,:3500]
caf.display_mask(filtered_particle_mask_cropped,10,10)
debris_mask= debris_mask0 & cut_mask4
debris_mask_cropped = debris_mask[:3450,:3500]
caf.display_mask(debris_mask_cropped,10,10)

In [ ]:
caf.save_mask(filtered_particle_mask_cropped,image_path,'.png')
caf.save_mask(debris_mask_cropped,image_path,'_debris.png')

# Identifying Chains in the Mask

## Below is a limited sample of the code that identifies all of the different chains within the mask. The full code can be run in "Run_chain_analysis.py"

In [ ]:
original_image_name = '53-connectors.jpg'
image_path = os.path.join(os.getcwd(), '0.2_20mT', original_image_name)
original_image = Image.open(image_path).convert('RGB')
mask_name = '53-connectors_cleaned_mask.png'
filtered_particle_mask_name = os.path.join(os.getcwd(), '0.2_20mT', mask_name)
filtered_particle_mask = Image.open(filtered_particle_mask_name).convert('L')
filtered_particle_mask = np.array(filtered_particle_mask) == 255
caf.display_mask(filtered_particle_mask)

In [ ]:
particle_bounds = (10,500,0,500)
region_counter, chain_mask, particle_region_masks = caf.label_mask_16(filtered_particle_mask, original_image, particle_bounds, disk_size=1, connectivity=1,
                                                                    branch_length_fraction=0.007, global_min_branch_length=2, min_region_size=200, debug_plots = False,
                                                                    prune=True, prune_branch_length=5, max_hole_size=20, vertical_prune_length = 4)

# Load Completed Labelling For a Wafer

In [ ]:
chain_mask = np.loadtxt("0.2_20mT/53-labels.csv", delimiter=",")
chain_mask = chain_mask[0:500,0:500] #Look at a small number of particles at once
unique_labels = np.unique(chain_mask)
unique_labels = unique_labels[unique_labels != 0]
num_labels = len(unique_labels)

# Generate N visually distinct colors using HSV space
hues = np.linspace(0, 1, num_labels, endpoint=False)
np.random.seed(np.random.randint(0,100))  # Optional: fix randomness
np.random.shuffle(hues)  # Shuffle to avoid nearby labels looking similar
colors = hsv_to_rgb(np.stack([hues, np.ones_like(hues)*0.65, np.ones_like(hues)*0.95], axis=1))

# Create a mapping from label to color
label_to_color = {label: np.append(colors[i], 1.0) for i, label in enumerate(unique_labels)}  # RGBA

# Create the overlay image
overlay_img = np.zeros((*chain_mask.shape, 4), dtype=float)
for label, rgba in label_to_color.items():
    overlay_img[chain_mask == label] = rgba

# Add black boundaries
boundaries = segmentation.find_boundaries(chain_mask.astype(np.int32), mode='outer')
overlay_img[boundaries] = [0, 0, 0, 1]

# Display the result
fig, ax1 = plt.subplots(1, 1, figsize=(20, 10))
ax1.imshow(overlay_img)
ax1.set_title('Watershed-filled branches (globally unique colors) with black outlines')
ax1.axis('off')
plt.show()

# Analyzing Chain Data

### Analyze the properties of a small subset of the identified chains and debris in the image

In [ ]:
mask_name = '53-connectors_cleaned_mask_debris.png'
debris_mask_name = os.path.join(os.getcwd(), '0.2_20mT', mask_name)
debris_mask = Image.open(debris_mask_name).convert('L')
debris_mask = np.array(debris_mask) == 255
debris_mask = debris_mask[0:500,0:500] #Look at a small number of particles at once

In [ ]:
df = caf.analyze_clusters(chain_mask, 2/3, angle_offset = None, fixed_angle = 0, plot = False, print_statement = False)
debris_df = caf.analyze_debris(debris_mask, 2/3, angle_offset = None, fixed_angle = 0, plot = False, print_statement = False)

### Single chain example

In [ ]:
ex_mask = np.where(chain_mask == 20, 5, 0)
caf.display_mask(ex_mask, 10, 10)
ex_df = caf.analyze_clusters(ex_mask, 2/3, angle_offset = None, fixed_angle = 0, plot = True, print_statement = True)